# LVH × filter bandwidth — does the 40 Hz low-pass break the LVH head?

**The question:** the synthetic-ECG pipeline band-passes every sheet at
**0.05–40 Hz**. The 10 AF+LVH test ECGs that the frozen four-head
**ResNet-50@768** (`resnet50_hires`) scored as LVH false negatives were
regenerated through the SAME pipeline at **0.05–100 Hz** and
**0.05–150 Hz** — everything else identical (the regenerated 40 Hz control
reproduces the released images pixel-identically). Does widening the upper
cutoff change what the frozen model sees in the LVH head?

**What this notebook does — nothing is trained, re-thresholded or re-split:**

1. unpacks the harness (the same trimmed architecture the previous rounds
   verified — SingleStream + FusionHead, four heads);
2. mounts Drive and verifies the frozen checkpoint, the frozen
   `test_predictions.csv` and the regenerated experiment images are present;
3. builds the stored **768×768 views** of the regenerated sheets with the
   EXACT transform the released 768px corpus used (RGB, BICUBIC to 768,
   JPEG q95 — no resampling at inference time);
4. loads `best.pt` STRICT and cross-checks its recorded identity
   (protocol fingerprint, views, threshold vector incl. **LVH 0.47**);
5. runs all four heads over **original 40 Hz**, **regenerated 40 Hz**
   (control), **100 Hz** and **150 Hz** for the same 10 cases — the LVH
   head is the read;
6. cross-checks the live 40 Hz probabilities AND decisions against the
   frozen file and refuses on disagreement (tiered: float drift vs
   JPEG-encoding drift vs mismatch);
7. prints and writes the per-case LVH table (probabilities, decisions,
   deltas, flips) and the per-band LVH metrics (recall / specificity /
   precision / F1 / accuracy / confusion matrix at the frozen 0.47
   threshold, with the frozen LVH-negative pool as the negatives —
   declared), plus the verdict on whether a higher cutoff improves LVH
   detection.

**Verdict rule (frozen):** descriptive on n=10 positives — SUPPORTED if a
higher cutoff raises LVH recall on the probe (≥1 of the 10 frozen false
negatives flips positive) and no case flips the other way. No significance
testing at this size; the probe measures whether bandwidth recovers the
KNOWN false negatives, not a population effect.

**Runtime:** GPU ~10 s inference + ~2 min view build; CPU fallback a few
minutes. One pip install (`timm==1.0.28`, pinned to the reference version).

**Drive layout required** (edit `DRIVE_ROOT` only):
- `cardiosentry_runs/hybrid_v1/resnet50_hires/weights/best.pt`
- `cardiosentry_runs/hybrid_v1/resnet50_hires/predictions/test_predictions.csv`
- `filter_experiment/` — upload the regenerated experiment folder here
  (needs `by_band/0.05-{40,100,150}Hz/*.jpg` and
  `by_case/*/original_0.05-40Hz.jpg`; the 40 Hz control column doubles as
  the pipeline cross-check).

Results are written to `cardiosentry_runs/filter_bandwidth_v1/` on Drive.
The frozen `hybrid_v1/` run folder and the generation config are read-only.


## Cell 1 — unpack the harness (no installs needed)

Same convention as the previous rounds: the whole harness (config, model,
view builder, inference driver, analysis) is embedded here, so no folder
upload is needed.

In [ ]:
import base64, io, tarfile, os
os.makedirs("/content/code", exist_ok=True)
_PKG = "H4sIAHpViWoC/+19W3vjNpJonvUrsJyHJhOJlmzL7VZGOeO25W6f8W1tdWZnHX0cWqJsjiVSQ1K+xOv/fuoCkABFye5JJrt7In5JWyTBQqGAKtQNgLvhbvzp3H/8HPijIPnmX3I1+Vr2t9nc2i5+4/NWc7O1+Y14/OY3uOZp5idQ/Te/z2tzV0yzcBp0W+9332/tbn/48N7dbu9utTa3at+sr//vr2EcjcMbd/b0L6wDmXqH+b31vm38xZ+tdnvzm1Z7c3tzs725ud0G/m83m9vfiOZvyf+TMBplafD0eB+GVeVee/+/9LIsq3YYTrIgaVz70eghHGW3IozGQRJEw0AEj7MgAfEQZaIh0ls/CUYChgzQLMpSt1br3+plOiKD+1ZT7B1+d/zjZ5EFaSZ6+59SeO5n9HKcxD8HkRjH86RxC3NO7SJIT4Os0W7+6f3OrrD/lgRpFGTtpncbws+/OSIdxlirnwoEOfYnaSCi4MbPwvsgFQ+AaC0JboIoSPwMymW3STy/uaXKLvdOeiJ9iuB3Fg4bgImYhbMAujIQD2GGhZIgENjwxsxP01oaZFkY3aTCbrrNdmO7KT7/jO3NknhSF/Ss1cSH6qaNN44r+rdhKqDiaCQmsT9Ka1j94cXZf/ZOxfA2GN7N4hBoCBWJZB6lhB02B0kg4vsgEf5kItHJWxMCoWuzyZyLx0l4E0b+BN5PAj+FpoZT/ybAXpAVHRxd7h+dHx+d9oSd+tMAaTZLgvswnkvkUqdTqwnxrYji7BYaKhDrIEt8oMioDj8biEJ6G09GAD9O8Ek6m4TZ9/QVolG0pq53KHa1B5WNwiHh7Q7Te2oulsm7AKAIUbROsPQRMKygJn/UiKPJU1EVEmgUDMMUi+aIIc5axff+pJHNAX3oke33xccTGB6CO3CWxNf+dQjNCGHETIHjoTZ4OJrDCNcHZTiBwRCMYbwB6k+E7DCezvwkTAGDaD69ho4KkajQfKixIXwxje+RkEmgWGboR0BdADC8BQL6BAU6GAkjGQZbKYj+kjGm/pMYhWOAANVnDwHS8zbQRmdKJMJns/AxmBABamNiWyEbMvqe3i8S98HHPk4z4qIsFtrI9scIAGFpfH49Dycjt4aCAegyFZ43nmfzJPA8GHCzOMGmQQPl8Kz9QTR+8cWt9dM7gHaWgB4KFMkAtZQFSkFb4B0xT4nLoRtiKqoGGY6WYTyZTyMRj6lRfxCpf88iQQ6d+2AIdMAuBGkePIpNVxzEyAxQB0Fza8d7H3vHl6IrrqzLfu/kyKoLa+8Q/4UK8M/p2cXJ3rE1qB2e7X+59Kg8FKfXNfjHOzo96P0HPGFILtVk02tHaNcfRLcrNn9FAk7jUTABcJfE+snwNsyguXMcONC1UyAE9SdLqU1h3z5dJ+EohbnfQdbHfs+4CCiEU7f2cW//zx/PQJhA45RUdrN7L4xad1btc2/vwPt8dHAAgqcr2q1NfnJwcXZ+9qUPj5puCxt3WGbTJVzNfWb2tSt6wDlPACUv1xgFswAoCgNV8uN1MIkfsEuRVefIlnt9Hh9Q5zyow2SB8hVlG9bvUh/1P1/0Lj+fHR8QotvvEdN+IQmAGRoTHyBXYJpWoipsHo1dAMQdD7PCFxyqxO3XT/SJgtaYAE4TMUziNG2QRFVAjSnHlZK9QJcGJiC8jRPQbqtOyOO/Wx8G2IZ9DeIoCceZFB92GqBM8ydPPwfY4R0oK8QfRSto7MCPEAkaDmFykRdNH8CEtygxxjClZTiTMPLFp1vwA+fRMPK4MryAwYa3G8P5wemp/JLf2QW9iIXHs9aOU8DahB/wMh6BWMyh/d/z3qcGPQ2SjfNwgj3NghLhIBcT0UBzmD0SqOXXfRg8pN8LfzgMZpl/DZL+7PT4ryIc08zLZFfdk74CK80AFSTP8FafP/LPQSXyh9nkicD80JXNOzm6PNnr73/WwMQz0ZBiXaoloxiaxQJJm6FewUfWH+CAJ7lcO7g4Oux7H/dOecTY2M8gu/J+toDl8eEWPNR7EJ7XSsCx2CYUMzvHcgaA1B70cDgTpm5nzNTjiX9zA71tTf2E1BdLPNxCoTBLtXkZKZXio+EkTnFKimlg4+wYMS83ROgGLleDVQ7jOVRwjYMaZF0WjIHghdyPrwEKin4eSDAh3tJE5/N4kTPnEO5vArd2snfx6ejUOzze+0TioLmpxIHB4u9wHh0im48kw2RPdZ2HUUu9AVUqZXLQ5AOASvqfDfqFqZI4xmdKBBW6FCkmbu2id9i76J3u97yTswOec0xl2dKKnF+c9c/2z469w3MsuLXzYXd32A6a13579H5zpBf98aj3FxolwEYD7Xn/6OQEv225TXdz1/qVZqowAiGdj3eAeYL6WMFLplS9fsqAIkAw+mGDphqlwyS8lpMZUtcH1QoGDQjMJ09Oad59ayM3bJ2OkMoPiQkWBHVxeoZakT8F7RbUNz8DUKi549wHc8YRqtZglwArJlN/EqbEVi4NCg1MMRXkBgiO5NsgyFhlU8oZTgDQo6BKBh59CYgpEyQQvf/Y24c5CxsHbcVWqbIjH9SiIMPSLDTjhLXwOJmBUm9ffPpYFx+P9r/A/8g1gNsj/F8nySn+8aHtuDVUWLyT3h7O02DXbO+2adpo79CfJghhKnHZP+ACm5sf8A2Y4vyn7dSOz2Bqvzz6T9QFAHxt//Pe6SnMcaD9XOJc30/mQW3v5Fz9xCHl/fuXveOj/l/h2Yf2r6jmIOsEkj9LpuYQaGUYB6QSrbYg6wBKTiOaGQC8Hj9EuoSS2qZ9G97cYl3jMEkzoO7+3mWP1UUwL5vNra2tJmqJdLPT/tAubrZ3tvKb9u77neLmQxuKFXJXPm2/L8pvb25vFjfNzQJse3uneLP1YWfHIkXgI4qa3JwtD1NUaAqVn4c1T4HER9LEQn3i7LR/cXbcAXldzEipBGaYoWyZNPIZBpQeG/SucBwST5DQzwxvAVkmpHXCZJTGWMUonsPMzNKUa3mAiaSYHlEplVOaJe2Yzz9j65Vprt+18Y6I8SNYcD5UeBc8pcp+IC9BPJc+i0Xjmpi4XsHduTtAGR7ZQ6y1CmCQyuXWfty7ONo77TO2WIG3TUODwBm/AXHtpt0ErOXHNIUDgOccQEeYLc+hmS/Ks7heVadMr6LijkG8lxwNZeUYeOREwypRv9MHhFOBg4mt7rVhCJK0qz6V+OvfymYs/6S9+IlsXmGfoPxgVweY0PE0AC5YNbdrosKt9XuXfQ+l64etXb45P7s86h/92AMBCRIHzLx2+3+oN91dx3/W8R8t/vO+3XZ3tzZ3djd31vGf38FFGtK/NvzzWvxne2e7qeI/rZ0dkAX4aB3/+e3iPzSZNVhZLmI/ytYYl8NDmuZYq31Ed3GqmTGFAWf4IO0qJySo2YUXslO7BGV1ElxmSeBPxXficI7eFJyZ6hQwomhJCsoihVj0efgaZm93Buqa1Dg5/lIY3tWmex6NkRqsHjoiBxg6VGozPytbpn+Txrnt/I3UyWoLM4o1C7NeqzYp66Atoum7S+X7/b06OcaEP89iMGcyAZr6p/MvQOhTGbK5BSRIQeeoDUZpoOdIzxYN8RCAfQLGJ9myKdB8yF4HGOVPRYzl7e59KjOMJ5NA+iJkIXLTB6MDgM9lkEyT8Fq9P4fbWk3eRPMpGLCg7Ucz9Yi8hMaNG0VUJMo/U9GhVOzXfg0DUvOSDyd+mmoDzI4i9wSMm0ngdEiRBPr0Xxuw6JTKHadA6zmNdOpI6hzoLYI19YEPQPV8csU5MNINwJqJY/8pSE5hKIjGD9jSoZ/hr5Pjc/yzDR14E2api/1EQEbBGPoKgGaeZ6fBZFwXBMgbhdMU1FutQ64AlzowcjYA5R5o71FrAygFz2T7yP84Bz62HTcH6xSvoAKX4UfACSnosDBiM7uoslQWBzWWyumIiNjPNx18krfVHjkkVm7qYoRWdwHOBUExTW3npYCbxWhMdQHNqVavS159KFlCYDqZcfWXwT/m2Cn+xDZMAnh1kMQzMPnsfVePVZRcnogv9Jaf2IRAXcjSHOtYLPypd/zFXnz8lXUZlRi9pn1SanIYeePAxzGHlCdsi7ECZH7wk5EcKliucpTE138Hzh5Y2rCY+UmGAK+Kjr26GdgEAn7IHsTuK4+SQQ4jCQCtKO8ZmzkcxrhN0KH7w2m3BX1Yk6yoi/4qZrzI+U8vWVehrQ7ISQyVDu+u8Yef0T0QJp7MyXO3io8y/wYlOFAjDX8OiEtW8o2SWlDzm3mJ3YVdYWMV9dJLfIYDHf6YL/IGdTkmN4RGZ4FHMswGtOvoKJQTQfcQHVsG4t1mhVW89LqZxNf+xJvF8aRr+fc3VoEl9Bbip6PkYj1q+JlYU1JDV5eu2rCzr+yxlTxjW18sGgfOwDEH/LJB/Kj1gT7AbpdUoUiL9Ri4248OVIqDDyuhgLeXz+4FdWlQQEP2XRUKXUXOfOzQF7nHdNUn5TGGUjaIQBbICKJsLytmXZNHqPdpMJm0yz9wMbwRBZPUm6AegUgZblsqGY7Lj9mtxew6H/lumHr+vR9OMGRmax2gkOK6stieBtM4efLQge1nXQlBR4Fxkx3H0/CvMqsXah51J2o+XvHMRsWkQ/qIgzMr6m25VDlGLSlXHrN4Qs5RULLIr4dNwPDdpruj9CqPorlQjT+fZCRUaKZKngrCDO9IGGPzERWqvw46wMybgEaHwqhrDWdzGJM6zC76zZlCwSOGKEWP/kDxXwyaBIOj+huDi2EaUg7XMLCHd3WiiYOKpEXdYnGZCOrTGM4P00D8iJNvL0niBFjsGet/QV2UEl5YR2ps6gEva/noH1v2NExTpPY7qvYdKqlZwNhYxmgZ3kleJdBa73pKm4d2dGQ7oJNRV8H5bVDMH2X1X9ynecBSRpVmHOO5oDpTAZIHVTiBjuc072t8SrMjT3Zj1DqGd+5NkNnWLIlBa48n3hi+CxJKE7JywkPRf0MurAzSFYSmClx/hmkOQGQFU2gwxfN49oLAZAOWUBl6aEltLxIrNSupBtB9gTG/LiFN4cKl6PInz/RHR/F5AYTCAawuDYMizaJAA0vIQXZKsztm2YGchOcOJtEYMlObpsfCv05tykHAsldQRqXngBbTEHyf51Y44geM1H/oGNQstY9iTXmA+7kM9g29kveMUbmiBgjLYAlxLY2vhn6ShJQpsJDdZHIOQfh15GzuGVAqG0ZILoNCml7qRjAHQTuYFAHMjUzZj/tBlIKUgd+nbAPj7MWBWPjp8ryPsShpTuMowp6X6RQM+Xsjbhs9PbBBLPMEyFp/wOQAAnYNc+UdSfN3ZNFjGBm0zzy95Z2e4JfHYw3vvmzcUu0Ra4R2ksjBWWagDUHOpkKBfc9ZREp1VKHeFK3bvqk+waAl7TtL7Bkr2zNsHtUzMItmY4y4uvvQhjgN7Ku+q4gMFgm8yMkM/JGHgdGiURFfUIP0hgEnyXYtqlrIZQo/4yPgWzTecoKEZX2ZqHB+dKxaT56QYowj1emRG8+0Oq5CYFIfHQ4mQ4bTG2h0CMpwHN0HCciMi08fNV1V1wyzsQ3FnbqcNDMmDUKW8wlPJR4Ov9Re1pPL8Tc7p7O0TbPKljzUxS03xdD8QW49sNAtwu5Qz23pmQmLp+fLpxR6ovcYZnZtUeZIriJvF7LV88PL4/PtS50k67MGGh7rd52fIqsCnBDPs5dlr/o5M8p8W39UpGQXbrKKlI8qePi5K8jVSF/wLENUXsiqaJh5HZXwCicnxj5YXgh/BP/DuIFxNZIiiaaaqX/HubdVkPQMAU2UTAM/Im09Bc1BDTZonZfXbJPOUyE+pEfQ0+2J3SorYhTch8NAGSkWausWaT/RzI1GMEH4T4VoDm+mcTgqpULbUV1sO7mj9ya8B6mXLkpxdCbJuVqV4mxZzjWlPEE/LfVl7iPNVadCGrpglE9SF7NbFFMdwG9Ux4OEZcsiZzq1SruDyaAbPui2lfb3CEWpmqh0MBOqCl4W9dqjVKd9t/gJE8btfDyeBNLGrq0y6B7i5C5I0u4m6NVJPCPTR1fB52kA1hrUDHLTz0DIy4FgGcYS6PDaNzCcyYDDFBscktxm1H2425lk2LmaasrTGbtWY+8mAYNBF8rQ6Y914aH0YmKYAuURAD0W9EUhEXnXYGfcwbjWrBVNanHLOgvUYVAY3A9v5vE8faupaEhTLqXc4hIrL3uaBV2F4YjuuCBpfa2d1/0fQYSW7agLJHYWUWcvrBpu9qOJGBFcqWdcb8qMZkv3LSufjuOCWQacQI5w5bqUExVwK7t/AZMssAlmXfiPYdptOv/yGOw6/r+O/xvx//dNd/fD7m67/X4d//8dXGX16beP/zffbzdbefz/fXML+H9nu7mzjv//VvH/Qq+WSl9hfVdlOKv85kZl7vJrmct5liDXxYnMcmFc/o6cXjibMy6paO3s7jy2tlqtPLlaJl+atgOZGT7i0GAFfqQyoqEiwOS1fOm5T4o8pk2jH8JYWKolrmoJ3qCjxQV2lMeapw/kRCEbqIaLFWZglgCsj73Ds4uelmvRoAap16AOoM6ZL9kbhekdpaJG0p0yqskFRVRPnqEgs8KlbotkR28m0IGWekZpOGJDBvtGFmJXiWH64NrGZBZPOE0Y8foaL4rsyjSeJ9AujVK8DEulKnzP66IiMAHJDkRfC5d7SMIMdKG3pyioZ8kNoJAGefZAer8iM6Hapq/OPEDbLY5YRtppMuRoAuibaaZ+vi34I8dXUVDPkyfTDV5oNhsakExCfLeQY1K4qqlDcsrlruqyZyUZLvghlvsgpmFk41tH/JGb9/X+Bh4Dz1Dvi+FvSKf+ZKIvyEkwZp8il1XZ2BwshEY3MEf8FlrgT5DTn8Q1LpUdxQ9ROvQnONrPeZlNttRez3OWyxLF1VxIFO1c8DC5LFJsW5qE8K8jV4q4UqRoEVOMKcFwjjJ3ejcKE5tvUrKc6iBMwN734ruSIYXd4N8HYIlmYBCiTAI7UI6arvxbF/EMNDa0UIuPpS2BtWIQBUwNAE89qsc4geo2d0rKwZKrbD6bBJwJgOGSupAuCJL9XhLHaoTX3hTszCNsSwDnYzvH4tnGRRtQIafo42IdTXIIjjAB0IVyNbmCzmQJ/oCk3nyKhl2HSY0CkOczXmtOslHxSTzHZjy/5C69uwCoDKOWchxAHgQjRTaVpqKZigZeAAc+1kcBPCmIKTaoOPyBUY2/XjzvWX4KP7EgtMX9++zG0lkR/XPYtdSU1C7ZqbpoIqEkx2ZeCNp3BVgNABV4q48XeCMHyNQHdjdFkI8+ByVV3b3kZo6T3zneJWB/5xGzrueN4qHnSVfFzIXZxPNlcdtqNACvBrbeqhOjh9Blkg3IYK8aX3zdBpNZ1ypyED1tDh7jHgWJsK+fPMxM3KiDEPSQpBsqdliBCbT3l2DCoYastAZMm7RUzckN+gwAAaIdogC9xk4aiS+TNnWBNmpkWPKVpcrRUKkuh68sPaIrv8VwPUobiubK51g2f14O6GoSHIakUdULidooVrA3BK3DYyKDLF4Z3wL2j6XLl5fwLOtBco8SL79BMJlsSvQBFt13aRFW0TQU8lBUkTBnulz6e/lKmUpew2JLmK2CaiqK/YyfgbASsu0LLd5YRTImG+pRkwCXOipCu67r6HOTJNOVlIf5ehwHiYM3hodPypY6rUlCWtm2sf6oWDHkvOYqs83FStoKord/2m6Wlmc5JfKi0O3mPLIh5J9cWvpNs8O0ToNPl/TZa/1G+knebdbrKVNjq6Jvja4jfjH6raLv1FyGPQc41IyQ/MKMXWdBACKMuNORLl9McRlbtIxug+c18YyhOoLjvFRkJtN0asBSwWccMSXc6hzfkrMgewkW58BfFO9SbRCCOrnTaqYvQs2K8o41R/E8K6k3Gxut5ub2i/jzRzPs3YRpLcTgJOYlgt2AnnLPw0nO8yyZVUUz3u/dybX2/679v5r/t918/9593261Nne31/7f38FFLqj/1vVfzZ2t1na+/9/Wzg6v/1r7f3/r9V9aelnhmKS96eTGRIZf2G415X4Lj2JbqQu4v5y5KpzN+O+EudgbZm34v50XSM2FZNVLyNS6L5mvKy77F0f7/WLNF21ZoDciX+hFntN83718aZfc4Kg6a6FGqpS96BEvL+xasoCL1205bu0vYBoCcrgfnccB479jhgjqWsWGTKB+Vm3Jpy/u+sUOUaxW/U6f0teXbfEarSqfqHzEuRjw5OQ38CJAt1bY7WyYz+ROVXJoLHcAcIJrGcoKkx+MZgyBVGjRTL5S+Gy16wF7YFkT0IMQlEfJcnCc84ALKDgXvWthasQb/A8qdYStCb5TjgQtrYSgFep7/hXnm1BW7rI1ApTDKigRvQIwf49mHRptb1howOaBtf/lYE/Mo7wIcMwYrCPkGFzWgb2/f/5F2OkkfsBd4eZZEZ9w9KUsRUsQwcLmYvdKl1jAJuKwCcV2VjlLS8sAX+WJ0Gxwfq82BTEto9kv9hJqhvDsnzODNRnX4cw+sSQfT1SlwZWy8Gh7nJL5S0RU6TIzp7ZgwY7IHC+W27wIG5c9OShMn7njlKmq0sBO3OrlO7p5bOG8gaCLyaFjSXMapQqXpcUVJ255+UgxHKionmdGRWnNgkdzxvDuSi6gGNTl2lvNRS9xEdoqB47UwSzDhUFwAlpovGuQwAK3cN8a2W61/ODEXbUUojDnH3DY0UcLHCXEX/YuTo9OPyExHvJs++rCP0WX/bPzjqiYX4s96xZ2xCuPHctYaoHxnOgeRHZ4gxSRO6/muoe7mNS7aY4ZGJ60QoMcHpzc91LSUp55cQCxJbxUGouOGDAZF1K8CeVyfUa4rsJDpdeduMsTOY0Ewt26lDZdlaPIqYe4My1upUMALWC2Ifx/3+HFDSQxTFnxwjJGEzAv7AQNcbu6Qggtl0CrZA+icyVxGVwNB1f3JNw4c+2Rs0MfKccZi1yFg4EpdMR3XdGqFZDkzETRhmJ24XdyTAtzRQltZ1czcFlcRjN4fZ2NFswpuBZ9XDiVOurtV4bl8Auam70seMxsBOWO5tNZaiOyddpINsq6m47B5Q+AYYB8BZ8bgg4YCZeQlPJwizUcuO2h07FM754Qz+9w5L5jb5gF+jQ8s9y/Ax+i2/6+8wO+sKo623FeGSBJ/ICzoQ6PuuEd9fc7OSSMVS6Ahrs9flk+M1ThUenvk+49wOHlqx14oMPiTJfZrHU6a3/F2v+39v/9Uv/fVtvdbe++b/6P3bFsff2aV+GC+O/y/23tbLa3Cv9fC/d/am+uz//4Df1/5Q2eUEOhgZHSzsv9zz1Q5VApSMjyppRQdd6FW6tdBGo7JlJtcfN0XmNmajmaXaBZEfkOuXFUW5Joape8Io6xM2O6uG96Td+qUR6OUXlABe5JPkeLgBId8RCG1L9JgoAPQgie4mhUI0UY9MwG40s7SjvSp0iqDCejko6Y0vEaLdfY2Z13G9e0bfaMFuvPMfexjmQTtOc8QNh0iYrkBaAVxxoIecTKou9VqLUj+T7kr8XloVMzHz7AHbXJ0bHlcsQd978Mh6npOghwV9eNdAbQx+EQDL+NfEv4jcPWhj8czhN/+CS+e6VWdCTS1iBoLybhI7Ucq62XV+C+Aggo01D7+ArctkQtQiuOh7GBFhM8tQZbt837h8vDOEqkQBvWF7S7byLmM8RpOM/i8Rhdo0l8Tx3xKj0z3iZLxHI5cqvJnUibFv8fleWsnM18gkEq9o7/svfXS8wgCOQiudIiZFoUTUMHPTKYOYspu+lCZrYaCt8v9SabR7z8Kkm2unv5n9kKbAZd79NC6NmoOgP38Kh3fHDpoeFCm+lS9lMdvQnzwKNjOcodYzH18CUJDixtPApGFR9h6GK7mX+Q32Lh/HYYJzDss4rPqSe074t7BqDul0PAL1WCDX6lfssv1C3xLd8sAdHWQLRNEG0dRLsKBIqDGXyZxd4sTkPkJFVz5avXgCh2rABSvKJdj+WibMTbnnX4+AgznCDNQ9rYBfdmKe/hIEGQa447vLwBzGzk4prPwwQMS4Y5wlX08Bhzij0Y0fSF9KSnaQCjEf1DozFtNrHv8na/dcwC0s4Nwly5Z1nuBY3qtE5LBobIl8/qK2kzR0EwIhcQ70pojmX5mw+doe6Uj6k78feLjhvB+iPuzpRB3S4fgYO7JJr45dlOVL5RLi4RS+fX6LcZX8F/CrmBi9vF5E60QZkw8I22DQcXAmxpT5jJpJirfLVNgyHbLB0cwroqSDFAsC3HxSwoBOmzHJXudvY5qsLd1jJIQMg3A9o7VHBURvVY7TyDMzq7W21Gv2MMpTrJZ20PGmOjoWLat/P4qBEUdQp9QFMF1JYYD7EAFSrJ3bKgYegLz/mUi/wrYU/9x/8iDv+vurg4uexVnTBRM5exwpDAORgX3wCcfVc7QeR7WRRPS+rtH10enZ3qCk6nfN5WcQ6Kr6tgWoX58TraqTq0T0CASe7lYzC8b+XBTsXBKoanCReWPGk96orzBeKgZoGuQb840YeP/yhAyYO7pjjXozIozxcqVF3cAR/mprTYE4QwEGrHFxd4ypPHPUnecaoTzLWQkF25NXxpofZdHXsWiweAFY2Yqh1vSJjhLogzl5b/21fkyUOaDK5knYOru0FVzGrgiJ8W1JuGGLuTeHgly9TlhgfeM2DzAoKB92I0s9Hn2RW6JFW4iop6MCB99C8jrcixjChep/bIceGVvpnjchjJNA1MCOk/ksy2R+Lbb8UmAAp8dAOaoP6w6sylJefWmFs0enTejHK0673CmZLEZ0jIn8OZnRNqVLWYnMwY2kqzqldq1cf60BfcZO4MrtbsCWfpxwnNNORxLX0NjLX0a2lx5d9K1GnCXTiTCnBfhBCONSD5pkh026lUoyWh82jC6sGgepU+4oHFvzlKFie08gLH1r3iIHJIoztf5q8iinduEI1SFA62pQapHEG0n2apHrkbaY7f22GX8HVywXBlEbI6h9CD4n1FU6tw0w8aRIe+Om7KyqUO21joUGe+10S8sWsWU++PVN7sqwI8QjFe0RrBfOMwVU7HQ+4sWEVV4K6iIq0N5plTnjxMyipoI8sSUeTv6vUtMOVQPr1HpvTbZvBFTfENsh41vzcmB8yapizIZwIz2lJEasrl8+liyQczPKy06hNUxJd+017yTXs5YtBmFdd/NkYFm2odFpW1UihYankd0Sq9KhtvnQr5V2jGA2fV5zBcOgvir1ClFz7WbUBV76zplMzBjjJSmku+VzZeJ7e8KGkm/0omyRyd7p9dXPT2+yULyrQlFR4J4WGYlQqRZAGRsrVZiUnyOia5TZpTA24c3UDNiUEvSv2vWa7VtMCPXsOhMHh1JHCrp4UKc/u3KNnW0G3r6LaXo9teim77zeiWkKhEd5mtzWO26CQUkSRAdaqRUfMWgO03AGy/DtAw5CsAtqowbL4JYPsNANsrAHp8vqDEjNR1GxVMOU7KbgLc96qcO/Umasuut1ah0C6j0P41UGgvR+HFCJ3r05aNojnf6BjAedK1+8/OgGD1nEt3bZEbDCY+e/7lgEvBttVc7wt+Wm2nx8JVq0443tnd0g4OZ4+wsvGbuFn3HXoOpJe0ph1/SW4OVunDRD2kYyQLg7QhzyDD0yJpdXeU8eGugUy4kKdlp/Nr3qafj83mLSxs3v2gODqOMwzLp3++c1zVvCc6n1ye55nb+rRLxOH5Rv9U2HIfi4RznzTnuoMEIThHB73T/tH+3rHwyXiRR7peP/FJ7HjsAFQrN1Ton28cnpINC8a69H3TBn8E6jYcjbDTkQjUheaWdSltJU+H7MmeFLhnRF083IbDW3SUPNz6mdwJjvblzBdzsjE9Mk1joFSuMF3JPyXvTnPBnQQfGY42Yp3FU7WK3XdRauBXV9qsLv09qKwrdT6Su2sTfDBsZ7Ul+hqrybqJbi85cm3lwke7+pC15hs/ay+czeYYSbEBbW13xdKBtTXNmjS3ui0n/2jantoybiWEFQCyGRiTkbSLCCuHUxj5Nzoa8xc1M5SEdsxMbAgb/v0OgOgZQspHYxaZFUXGLTxWDc96p/fyF5YpwcrDUl0JJYscDSKV5qfFEtuCCxGBiIpHJQRW6Ly0iLvD48h8I6kLL9XCXPN9VMzZ5M6CggUty/rzDN4i+a0x/hrTrwh/RehCxl9ZtKATIuG9NIioFmhgPlPxK5yUSt9o1MgLa88qvsi7Ly+fP6koPW7lxcativeqA4s5VT6oKJuHFj0OLSI1rKur/nn3OZu91EHodnHr7EFdXB2ews8InvXhRxa9DAblOR39SUr/lvgpxxO5moh3HKcCDbY96KDRXMVJuWeW0tYDGY39rZz2C5NmXTtsRp8Ivl4PkIayPRqTSVzWAeAxWf7mYwNxql7TE2AW6sgEzijQxGlCwQTemUAzmIlBGNdrtMlzeZCQjVaWtgtjtiyrcXt0TU7nIneZcr1UAJeF7zJl2jG2Rx7mmDM6y1GW5eN7OW3JHriCBgz0yYopQxnppWJuEswm/jCwLYUQ4qkGiuUsgGFbBCnU1e0o1DapWwoflVzML9XLwoYpjnxAhhjl3kiFVV7DQLlgtXNDEhkQx8Oqce1BSjVLSD+gjsvV0aqNtKiLN92X8rW2sPEwov7SURPJszaKOu7WmHeMgXu+sUrf28/UBy8boLdKbjs8Tflc8SwmzlPErSPs4d0Lv8SfaEqW4GFr+CuQCNIf/MxN7HyHibLCLnJrnnOavFRsUqRvKBzKvBRMu8HzVBVzD9mro/OKpmQsGXn4WXvhs/Zrn6lT26s0pFVcp5tjX8VtuhHllKIgXp2SlUOMysldTbIgQclWXuoCQwweE1uh79li36N0osgXuK/Vvqud/b7ollatL7b/x0zlK87FHrz8iYfhG1bI4FkXBItSnan6F6c44EDV06ke/pbWD+wwtYe0Y/V1wNEt3sG72CfH6SzD6TuBhOZEbwVT8qtMcxnJ0cXD7AddQJvF2qpYu7LY2AAIvad9uODr9bC9qKFZl1/Oz88u+r0DUSwOvA+gQSg91MksRnrOcupbsko+91vKisrMnPzwBRw4ksG6xtxEZwVKHjLerGjL6ZnAo3U+9djgkrlFjHYKspGXwOXxxnhlW6SdjS3JLc3KMyMq0OiLpWRdgtgKROhsmCIl6i10NUbz2Pqxd3FwtN/HRS8ari9Vha3Kh8fhNJSZSXVeToWdc+MnI9xwHQkpAXdE1AWi5X6JelXDoEVg5t5EqIj5tCd8QBuG1TXnxG0wGam5ws84zv19JbD8iPrCoH64DdDEF8UkQPMPZn5h6bsIj5qvglU+ql6eszOLZ3O5O2MwHsNk4pprJ6yfIsniRDbnN1icy8uHlixtffOiVibw1yzPHRend5eTO1cuxV29Chc6bT7JUjpq8w1LauWCLloaREvUtUWDbCRwahEtIHKcfLHV6xEi6RbBXJ0iIEPeEXykLa6pl4L3auFoJ8/7ybNEn0ufylwgDf6LXBYrids1sqmKpvEDOXuwQdKtSJNhp6Lcjn+sNjWrjMeVisp90irclnpBJUOQukUUUGmnuTkjc4aXrxIrlnu9YWUYewvZqImJRWxrFKagmj+5xOIwwDabTWdVuan/6Mm8Fii93TTXiXUt8a14v2usKtu/OLu8bOx/7u3/GXqtKpV4BSNUwf3aXBRt6Zban+lDisfrFcvHysMQo/XimWh/NX5XnRfybtBxN4PSejLETSa9FAfilZbFcaS6yHbCLBuu6p0eVUf4W8GLQNlXWvkpS8tRA0rdwtK7PF+IVa+C1pTZIDuwk9drhrYLgPEdisAhJ/7ZXFgLX4dVyc62lZv8SGM000A1JpUP783ouFUpJ5eF0B2HVBqJx2KKgcp3UTvi5dgvXRlcDMd8PTDYqpzdX076W1wYbJt2z97pwcbZRaHpGFupyYr7xhHPG/LIO0xKJ1RZ7I3iN65Mxg1B5r5cliy/ypK5TJkrOct1bGyUJRvCuuzv9b9cutljZjn6ilHT47P3kdWwBpNKLlioIpk2yiQz/xRVrYiWvbQwouQZfqvGkFPRmUKAvgga62J+oX+NKh8ZUZqHFjda4AS8BZKa2YDcB+Y6EBoLMqevooc5RES0P+gd9/cuKUiE3TBHlU/uaI2rpoPoPkziiEIR+XHXCwhBF4ej7yUjP9zGaWA0M8WjPoYTeJygH4BCR7RFcRkQHd+mWY4vhJdkLfFOGVbvVLorzXk00bmmMEZy23vsXADdWfvyAZeXID46giHHw8KoLMVK2Mg92xfOmNPOimuI0A1c3E9ZB2WZdiUYXaS45vs0s3kpgz2uYxlTlvPa9HXeu2hQDj+aDf29j8c9GY8r/DAljddY32K9VoHyCmSxhxsqRDc2+Rvl4TtfhytmRYmTHm4xdAlYIsZ0rK2ZMiiHiCTqWxAkn9syBKsY0co1GDdEz01zcLXoNh58XeOkAfYqvrpqJWvgcwR5tRPzPa8z+drTBHV1EOmBafdSkJqqIWkwvMz+UT9GSaOl9q2uKy77UpXVVFVplyxZ6E8FjZX+0tqilSpdcZVzEB0POS4vp2NzkM4li4Ks3fRuQzQLpS7rsOlMXFDworV3+B0CQ7ciMUURDKDgOKXUa1O+ng4wLk7R7NDI/aFbIQtk9cBek0Y2jyhMTFv0WzX94DV+5xjgjeRv1sFw9ieXva6NFbpS14C6SkdDz+2iUlbd0EKikDRhVlkhApZA0XldY7evgqFYqm7YI1xgYAw7HjaLekJhtXMJNczS+RSmBYxEFmFGuZFGZ3EbDQ0l7ShVUE8tTrUrDwOtvMYPmLFHg75WDh2qPaoBYnLFgZxBRyRXq2MeS7aIQIeudOeygMvduTpe42h5rePIGnwNKJz5l+RaqXDG0qiRCq2shtd+G7x2JTylvXWMUeSCvRgWYrsjLOeqNeCn5OGxHZgWGMyLMdbk2Fkt3mShha1MXlNsrYOz016ukyqD6aeI9j4hPgYAX7+tx8IWVevtPdbX+lpf62t9ra/1tb7W1/paX+trfa2v9bW+1tf6Wl/ra32tr/W1vtbX+lpf62t9ra/1tb5WXP8PfvdIxADIAAA="
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(_PKG)), mode="r:gz") as _t:
    _t.extractall("/content/code")
print("harness unpacked:", sorted(f for f in os.listdir("/content/code") if f.endswith(".py")))

## Cell 2 — mount Drive and verify the inputs are where they should be

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# ---- Drive layout. Edit DRIVE_ROOT only; everything else follows. ----------
DRIVE_ROOT = "/content/drive/MyDrive/CardioSentry"
RUN_NAME   = "filter_bandwidth_v1"

HYBRID_RUNS = f"{DRIVE_ROOT}/cardiosentry_runs/hybrid_v1"
REFERENCE   = f"{HYBRID_RUNS}/resnet50_hires"        # the four-head model, READ-ONLY
CKPT        = f"{REFERENCE}/weights/best.pt"
FROZEN      = f"{REFERENCE}/predictions/test_predictions.csv"
FILTER_ROOT = f"{DRIVE_ROOT}/filter_experiment"      # upload the regenerated experiment here
RUNS        = f"{DRIVE_ROOT}/cardiosentry_runs/{RUN_NAME}"   # THIS round writes here

VIEWS   = "/content/views"        # stored 768px views, built by cell 4
RESULTS = "/content/results"      # live_probs.json, written by cell 6

os.makedirs(RUNS, exist_ok=True)

print("Drive layout check:")
for _p, _label in (
        (CKPT,                            "frozen checkpoint best.pt"),
        (FROZEN,                          "frozen test predictions"),
        (FILTER_ROOT + "/by_band/0.05-40Hz",  "regenerated 40Hz (control)"),
        (FILTER_ROOT + "/by_band/0.05-100Hz", "regenerated 100Hz"),
        (FILTER_ROOT + "/by_band/0.05-150Hz", "regenerated 150Hz"),
        (FILTER_ROOT + "/by_case",        "original sheets per case")):
    print(f"  {'FOUND ' if os.path.exists(_p) else 'missing'}  {_label:<26} {_p}")

_missing = [p for p in (CKPT, FROZEN, FILTER_ROOT + "/by_band", FILTER_ROOT + "/by_case")
            if not os.path.exists(p)]
if _missing:
    raise SystemExit(
        "\nMISSING on Drive:\n  " + "\n  ".join(_missing) +
        "\n\nThe checkpoint and frozen predictions live in the hybrid_v1 run folder.\n"
        "The regenerated images must be uploaded as the WHOLE filter_experiment\n"
        "folder (with by_band/ and by_case/) to:\n  " + FILTER_ROOT)

## Cell 3 — install the pinned reference stack

`timm` pinned to the exact version the reference run used. Everything else
(torch, torchvision, pandas, numpy, Pillow) is already in the Colab image.

In [ ]:
!pip install -q "timm==1.0.28"

import timm, torch, pandas, numpy, PIL
print("timm", timm.__version__, "| torch", torch.__version__,
      "| pandas", pandas.__version__, "| Pillow", PIL.__version__)
print("CUDA:", torch.cuda.is_available())

## Cell 4 — build the stored 768×768 views of the regenerated sheets

The released corpus was read by the reference run as a **pre-resized 768px
copy**: `prepare_dataset.py` turned every 1686×1311 sheet into a 768×768
JPEG (RGB, BICUBIC, quality 95). This cell applies the **same transform** to
the regenerated sheets and stores the views on local disk, so inference
reads exactly the kind of file the reference run read — no resampling inside
the dataloader. Resumable: re-running skips views already built.

In [ ]:
%cd /content/code
!python prepare_views.py --src-root "$FILTER_ROOT" --out-root "$VIEWS"

## Cell 5 — build the model and load the frozen checkpoint (identity check)

The exact reference architecture (SingleStream + FusionHead, four heads),
weights loaded **strict** from `best.pt`. The checkpoint's recorded
protocol fingerprint, views and LVH threshold must match the frozen
transcription.

In [ ]:
import sys, torch
from pathlib import Path
sys.path.insert(0, "/content/code")
import config as C, model as M

device = "cuda" if torch.cuda.is_available() else "cpu"
print("building", C.BACKBONE, "on", device)
model = M.build_reference_model()

ck = M.load_checkpoint(Path(CKPT))
model.load_state_dict(ck["model"], strict=True)
print("  state dict loaded (strict) -", len(ck["model"]), "keys")

_warns = M.check_checkpoint_identity(ck)
for w in _warns:
    print("  WARNING:", w)
if _warns:
    raise SystemExit("checkpoint identity does not match the frozen "
                     "transcription - investigate before inference.")
print("  protocol fingerprint", ck.get("protocol_fingerprint"),
      "| views", ck.get("views"),
      "| LVH threshold", ck.get("thresholds")[C.LVH_INDEX])

## Cell 6 — run the inference (4 variants × 10 cases, ~10 s on GPU)

All four heads are computed; the report focuses the LVH head. Same eval math
as the reference `predict()`: stored 768px views, no resampling, ImageNet
normalisation, batch 8, no TTA, fp16 autocast on GPU (CPU fallback is the
same math, just slower). Writes `live_probs.json` for the next cell.

In [ ]:
%cd /content/code
!python infer.py --ckpt "$CKPT" --views "$VIEWS" --out-json "$RESULTS/live_probs.json"

## Cell 7 — THE ANALYSIS ⚠️ the deliverable

Cross-checks first (refuses on any disagreement), then prints: the per-case
LVH table for the 10 cases × 4 variants, the per-band LVH metrics with the
frozen negative pool, and the verdict. Everything is also written to
`cardiosentry_runs/filter_bandwidth_v1/` on Drive.

In [ ]:
%cd /content/code
!python analyze.py --probs "$RESULTS/live_probs.json" --frozen "$FROZEN" --out "$RUNS"

## Cell 8 — the verdict, isolated

In [ ]:
_report = open(f"{RUNS}/report.txt").read()
_i = _report.find("VERDICT")
print(_report[_i:] if _i >= 0 else _report[-4000:])

## Cell 9 — morning check

In [ ]:
!cat "$RUNS/STATUS.txt"
!ls "$RUNS"

## Where the results live

```
/content/drive/MyDrive/CardioSentry/cardiosentry_runs/filter_bandwidth_v1/
├── STATUS.txt               DONE (or ABORTED with the reason)
├── report.txt               cross-check + per-case table + band metrics + verdict
├── per_case_table.csv       the 10 cases x 4 variants, LVH probabilities/deltas/flips
├── band_metrics.csv         recall/specificity/precision/F1/accuracy/confusion matrix
├── cross_check.json         live-40Hz vs frozen, per label
└── summary.json             model, threshold, recalls, flips, verdict
```

All on Drive, so it survives the runtime being recycled. The frozen
`hybrid_v1/` folder, the checkpoint and the generation config are untouched
— this round only reads them.

**Read in this order:** the cross-check block → the per-case table (the
Δ columns are the experiment's core read) → the band metrics (note the
specificity column is held constant by the frozen negative pool) → the
verdict → the limitation paragraph.